In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../../'))
sys.path.append(base_path)


import numpy as np
import tensorflow as tf

import project_images_loader as pil
import utils.functions as fc
import utils.nifti_functions as nfc
import utils.util as util
import matplotlib.pyplot as plt
import utils.visualization as vis

import discriminator as Discriminator
import generator as Generator
# import feature_extractor as feature_extractor

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# tf.config.set_visible_devices([], 'GPU')
os.environ["CUDA_VISIBLE_DEVICES"]="-1"
tf.keras.backend.set_image_data_format("channels_first")




TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.10.1 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're using a tested and supported configuration, either change the TensorFlow version or the TensorFlow Addons's version. 
You can find the compatibility matrix in TensorFlow Addon's readme:
https://github.com/tensorflow/

In [2]:
input_shape = (192,224,192)

inital_lr = 1e-4

D = Discriminator.create_discriminator_model(input_shape)
G = Generator.create_generator_model(input_shape)

checkpoint_dir = f"{current_path}/models/training_checkpoints/"
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(discriminator=D, generator=G)
# checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

matching_ids = pil.get_matching_ids()


In [3]:
def show_generated_image(id, G, img_size, current_plane=116):
    def fast_resieze(img, img_size):
        return img[:img_size[0], :img_size[1], :img_size[2]]
    
    img_hunt3 = fast_resieze(pil.load_preprocessed_image(3, id, absolute=True)[0], img_size)
    img_hunt4 = fast_resieze(pil.load_preprocessed_image(4, id, absolute=True)[0], img_size)

    img_gen_h4 = G(np.expand_dims(np.expand_dims(img_hunt3, axis=0), axis=-1), training=False)
    img_gen_h4 = img_gen_h4[-1][0, ...,0].numpy()
    img_gen_h4[img_gen_h4 < 0] = 0
    img_gen_h4 = img_gen_h4 / np.max(img_gen_h4)
    vis.shows_3_images(img_hunt3, img_hunt4, img_gen_h4, current_plane=current_plane)


def generate_image(id, D, G, img_size):
    def fast_resieze(img, img_size):
        return img[:img_size[0], :img_size[1], :img_size[2]]
    
    img_hunt3 = fast_resieze(pil.load_preprocessed_image(3, id, absolute=True)[0], img_size)
    img_hunt4 = fast_resieze(pil.load_preprocessed_image(4, id, absolute=True)[0], img_size)

    img_gen_h4 = G(np.expand_dims(np.expand_dims(img_hunt3, axis=0), axis=0), training=False)
    # img_gen_h4 = img_gen_h4[-1][0,0].numpy()
    # img_gen_h4[img_gen_h4 < 0] = 0
    # img_gen_h4 = img_gen_h4 / np.max(img_gen_h4)
    
    d_gen = G([img_hunt3, img_gen_h4])[0,0].numpy()

    return img_hunt3, img_hunt4, d_gen


In [4]:
def save_image(id, img_gen, checkpoint_index):
    img_original, affine = pil.load_preprocessed_image(3, id)
    img_to_save = np.zeros_like(img_original)
    img_to_save[:img_gen.shape[0], :img_gen.shape[1], :img_gen.shape[2]] = img_gen
    nfc.save_nifti(img_to_save.astype(np.float32), affine, f"{current_path}/images/first_results/D_{id}_CHECK_{checkpoint_index}.nii.gz")


# id_indexs = [301, 302, 303, 304, 305]
id_indexs = [302]
for checkpoint_index in [30]:
# for checkpoint_index in range(1, 2):
    for id_index in id_indexs:
        id = matching_ids[id_index]
        
        checkpoint.restore(f"{checkpoint_dir}/ckpt-{checkpoint_index}")
        print(f"check: {checkpoint_index}, id_index: {id}")
       
        img_hunt3, img_hunt4, img_gen = generate_image(id, D, G, input_shape)

        # save image
        # save_image(id, img_gen, checkpoint_index)

        vis.shows_3_images(img_hunt3, img_hunt4, img_gen, current_plane=116)



check: 30, id_index: 08202


InvalidArgumentError: Exception encountered when calling layer "conv3d_5" "                 f"(type Conv3D).

{{function_node __wrapped__Conv3D_device_/job:localhost/replica:0/task:0/device:CPU:0}} CPU implementation of Conv3D currently only supports the NHWC tensor format. [Op:Conv3D]

Call arguments received by layer "conv3d_5" "                 f"(type Conv3D):
  • inputs=tf.Tensor(shape=(1, 1, 192, 224, 192), dtype=float32)